# 02 - Production: CHGNet

There are no methodological questions here: the parameters have already been
validated in notebook `00`. This notebook loops over the materials and writes
the JSON files to `data/results/chgnet/`. It is intentionally short and repetitive.

**Separate environment** (`requirements/chgnet.txt`): the three software stacks
cannot coexist in the same environment. `mace_mp(default_dtype="float64")`
calls `torch.set_default_dtype`, which is global state: instantiating CHGNet
after MACE in the same session causes the first linear layer to fail with
`expected mat1 and mat2 to have the same dtype`.
Restart the kernel between models.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.paths import RESULTS, REFERENCE, FIGURES
import src.materials as M

In [2]:
import torch
torch.set_default_dtype(torch.float32)
from chgnet.model.dynamics import CHGNetCalculator
import chgnet
calc = CHGNetCalculator(use_device="cuda")
MODEL_NAME = "CHGNet default pretrained model v0.3.0"
MODEL_VERSION = f"{MODEL_NAME}; chgnet={chgnet.__version__}"

CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cuda


In [3]:
from ase.spacegroup.symmetrize import check_symmetry

from src.reference import load_reference
from src.phonons import ph_to_ase, relax


# Si: simple sanity check of FixSymmetry handling with CHGNet
ph_ref_test = load_reference(
    149,
    functional="pbe",
    use_nac=False,
    expect="Si",
)

atoms_test = ph_to_ase(ph_ref_test.unitcell)

sym_before = check_symmetry(
    atoms_test,
    symprec=1e-4,
    verbose=False,
)

relaxed_test = relax(
    atoms_test,
    calc,
    fmax=0.005,
    fix_symmetry=True,
    logfile=None,
)

sym_after = check_symmetry(
    relaxed_test,
    symprec=1e-4,
    verbose=False,
)

print(
    "Symmetry:",
    f"{sym_before.international} ({sym_before.number})",
    "->",
    f"{sym_after.international} ({sym_after.number})",
)

print(
    "Cell lengths:",
    relaxed_test.cell.lengths().round(6),
)

print(
    "Cell angles:",
    relaxed_test.cell.angles().round(6),
)

assert sym_before.number == sym_after.number, (
    "CHGNet relaxation changed the space group despite FixSymmetry."
)

print("OK: CHGNet relaxation preserved the initial space group.")

c:\Users\antoj\AppData\Local\Programs\Python\Python311\Lib\site-packages\chgnet\model\dynamics.py:156: UserWarning: Only FixAtoms and FixCartesian is supported by Pymatgen. Other constraints will not be set.
  structure = AseAtomsAdaptor.get_structure(atoms)


Symmetry: Fd-3m (227) -> Fd-3m (227)
Cell lengths: [5.461226 5.461226 5.461226]
Cell angles: [90. 90. 90.]
OK: CHGNet relaxation preserved the initial space group.


In [4]:
import time, json
from src.phonons import compute_phonons, thermal_properties, band_structure, make_band_path
from src.io_utils import make_payload, save_result
from src.reference import load_reference
from src.phonons import ph_to_ase

MODEL = "chgnet"
DTYPE = "float32"
FMAX = 0.005
DISP = 0.01

materials = list(M.MATERIALS)      # or M.QUICK for a quick test
print("materials:", materials)

for key in materials:
    out = RESULTS / MODEL / f"{key}.json"
    if out.exists():
        print(f"[skip] {key} already computed")
        continue
    try:
        t0 = time.perf_counter()
        mp_id = M.MATERIALS[key]["mp_id"]

        ph_ref = load_reference(
            mp_id,
            functional="pbe",
            use_nac=False,
        )

        band_path = make_band_path(
            ph_ref,
            npoints=101,
        )
        
        atoms = ph_to_ase(ph_ref.unitcell)
        ph, relaxed = compute_phonons(
            atoms,
            calc,
            supercell_matrix=ph_ref.supercell_matrix,
            primitive_matrix=ph_ref.primitive_matrix,
            disp=DISP,
            fmax=FMAX,
            fix_symmetry=True,
            logfile=None,
        )
        tp = thermal_properties(ph)
        bands = band_structure(ph, band_path)
        payload = make_payload(key, MODEL, MODEL_VERSION, DTYPE, ph, relaxed,
                               DISP, FMAX, tp, bands,
                               runtime_s=round(time.perf_counter() - t0, 2))
        save_result(MODEL, key, payload)
        print(f"[ok] {key:5s} omega_max={payload['omega_max_THz']:7.3f} THz  "
              f"imag={payload['has_imaginary']}  ({payload['runtime_s']}s)")
    except Exception as e:
        print(f"[FAIL] {key}: {type(e).__name__}: {e}")

materials: ['Si', 'SiC', 'AlP', 'ZnS', 'MgO', 'NaCl', 'AlN', 'GaN']
[ok] Si    omega_max= 13.104 THz  imag=False  (1.8s)
[ok] SiC   omega_max= 17.967 THz  imag=False  (1.29s)
[ok] AlP   omega_max=  9.363 THz  imag=False  (1.3s)
[ok] ZnS   omega_max=  6.539 THz  imag=False  (1.24s)
[ok] MgO   omega_max= 15.951 THz  imag=False  (1.42s)
[ok] NaCl  omega_max=  4.196 THz  imag=False  (1.71s)
[ok] AlN   omega_max= 22.536 THz  imag=False  (3.49s)
[ok] GaN   omega_max= 19.103 THz  imag=False  (4.65s)
